# Create Ontario Research Funding Awards (Government of Ontario)

Creates Government of Ontario research awards from the Ministry of Colleges and
Universities' **Ontario Research Funding Summary** open datasets on data.ontario.ca
(Open Government Licence - Ontario):
- https://data.ontario.ca/dataset/ontario-research-funding-summary-historical (Oct 2004 - June 2018)
- https://data.ontario.ca/dataset/ontario-research-funding-summary-current (July 2018 - Jan 2026)

One row per funded project: programme, round, project number, title, description,
Fields of Research codes, lead institution, approval date, Ontario commitment (CAD),
total project cost, lead researcher name. **5,983 projects, 2004-2026, CAD 3.07B
Ontario commitment** (local run 2026-09-30).

**Research filter:** kept Ontario Research Fund - Research Infrastructure (3,926),
Early Researcher Awards (1,370), Ontario Research Fund - Research Excellence (391),
PDF = Post-Doctoral Fellowship Program (246), ISOP = International Strategic
Opportunities Program (30), PDA = Premier's Discovery Awards (18), SRA (2).
**Excluded** (in the script): YSTOP / TSTOP (youth / teacher science outreach, 79 rows)
and PCA (Premier's Catalyst Awards, product prizes to companies, 17 rows).

**`funder_award_id` (section 2.1.1):** the ministry's Project Number, verbatim. It is the
reference grantees cite: sampled `openalex_awards_raw` stubs for F4320314607 carry
`ER17-13-081` (ERA), `RE09-077` (ORF-RE) and 5-digit ORF-RI/CFI project numbers
(`30259`, `36544`) -- the same forms as the Project Number column. Only one identifier
column exists in the source.

**Twins:** ORF/ERA acknowledgement stubs are split across three OpenAlex funder rows:
Government of Ontario F4320314607 (this ingest), Ontario Ministry of Research, Innovation
and Science F4320321886 and Ontario Ministry of Research and Innovation F4320336370.
This ingest collapses onto the F4320314607 stubs only (see overlap cell).

**Amount:** `Ontario Commitment` (the province's share, CAD); total project cost is kept
in the raw table. **Dates:** approval date -> start_date; no end date published.

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320314607`; display_name,
ror_id, doi read from `openalex.funders.funders`.

**Priority:** `520` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

**Prerequisite:** run `scripts/local/ontario_research_funding_to_s3.py` (uploads
`s3://openalex-ingest/awards/ontario_research_funding/ontario_research_funding_projects.parquet`).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ontario_research_funding_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ontario_research_funding/ontario_research_funding_projects.parquet`;

In [ ]:
%sql
SELECT program, COUNT(*) as n FROM openalex.awards.ontario_research_funding_raw GROUP BY program ORDER BY n DESC;

In [ ]:
%sql
SELECT * FROM openalex.awards.ontario_research_funding_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320314607 must resolve to exactly 1 row in `openalex.funders.funders`.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320314607 missing from openalex.funders.funders') AS ok
FROM openalex.funders.funders WHERE funder_id = 4320314607;

## Step 2: Create Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ontario_research_funding_awards
USING delta
AS
WITH src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320314607  -- Government of Ontario
),
g AS (
    SELECT *,
        TRIM(funder_award_id) AS award_key,
        TRY_TO_DATE(approval_date, 'yyyy-MM-dd') AS start_dt
    FROM openalex.awards.ontario_research_funding_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    g.award_key as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'CAD' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.program IN ('PDF') THEN 'fellowship'
         WHEN g.program IN ('PDA') THEN 'prize'
         ELSE 'research' END as funding_type,
    CASE g.program
        WHEN 'PDF' THEN 'Post-Doctoral Fellowship Program'
        WHEN 'ISOP' THEN 'International Strategic Opportunities Program'
        WHEN 'PDA' THEN 'Premier''s Discovery Awards'
        ELSE g.program END as funder_scheme,
    'ontario_research_funding_summary' as provenance,
    g.start_dt as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(g.start_dt) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.given_name), ''),
        'family_name', NULLIF(TRIM(g.family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', g.start_dt,
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'CA',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    CASE WHEN g.source_file LIKE '%historical%'
         THEN 'https://data.ontario.ca/dataset/ontario-research-funding-summary-historical'
         ELSE 'https://data.ontario.ca/dataset/ontario-research-funding-summary-current' END as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (section 2.1.1): ids are the ministry project numbers, unique, no 360G-style registry ids.
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ids,
       assert_true(SUM(CASE WHEN funder_award_id RLIKE '^360G-' THEN 1 ELSE 0 END) = 0, 'registry id shipped') AS no_registry_ids,
       assert_true(SUM(CASE WHEN funder_scheme = 'Early Researcher Awards' AND NOT funder_award_id RLIKE '^ER[0-9]{2}-[0-9]{2}-[0-9]{3}$' THEN 1 ELSE 0 END) = 0, 'ERA id shape') AS era_shape
FROM openalex.awards.ontario_research_funding_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ontario_research_funding_summary' AND priority = 520;

-- Priority 520: direct-from-funder ingest of Ontario's own research funding summary.
-- Higher wins under the 2026-06-20 DESC dedup, so 520 outranks the acknowledgement
-- shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    520 as priority
FROM openalex.awards.ontario_research_funding_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.ontario_research_funding_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.ontario_research_funding_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.ontario_research_funding_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.ontario_research_funding_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.ontario_research_funding_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.ontario_research_funding_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.ontario_research_funding_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement stubs for F4320314607 (these upgrade in CreateAwards dedup).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as collapsed_onto_this_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.ontario_research_funding_awards t ON t.id = c.id
WHERE c.funder_id = 4320314607 AND c.provenance != 'ontario_research_funding_summary'
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ontario_research_funding_summary'
GROUP BY provenance, priority;